In [1]:
'''
Note- THIS CODE DOES NOT RUN AND IS NOT OUR REPORT
BeatNet Convolutional Neural Network

-Most code from miniproject 3
- Lecture notes 14 "Some Practical Considerations" heavily referenced
- AI used for debugging purposes and code generation where mentioned in comments
'''
import os
import io
import re
import warnings
import requests

import librosa
import librosa.display
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random

import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

from pydub import AudioSegment # AI recommended this import and its methods for converting mp3 to .wav files.

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [2]:
'''
Download wav files from mp3 data

- Combines folder organization concepts from lecture notes with miniproject 3 download function notes
- AI used to debug
'''
def download_wav_data(wav_id, data_dir):

    # Create new folder for data directory if doesn't already exist
    if not os.path.exists(f"{LOCAL_DATA_PATH}/{data_dir}"):
        os.makedirs(f"{LOCAL_DATA_PATH}/{data_dir}")

    destination = f"{LOCAL_DATA_PATH}/{data_dir}/{wav_id}.wav" # Destination for specific audio track (will be converted to .wav before saved)

    # AI helped with the following mp3 to wav conversion.
    if not os.path.exists(destination):

        url = f"{DATA_URL}/mp3/{wav_id}.mp3"

        response = requests.get(url)

        if response.status_code == 200:
            audio = AudioSegment.from_file(
                io.BytesIO(response.content),
                format="mp3"
            )
            audio.export(destination, format="wav") # Save .wav file

        else:
            print(f"Failed download: {wav_id}")

In [3]:
'''
Dataset class from lecture notes
'''
class WavDataset(Dataset):
    def __init__(self, metadata, path, transform=None):
        self.metadata = metadata
        self.transform = transform
        self.path = path

    def __len__(self):

        return len(self.metadata)

    def __getitem__(self, idx):

        # figure out which filename corresponds to the
        # specified index in self.metadata
        wav_id = self.metadata.iloc[idx]["song"] # Changed to song
        wav_path = f"{LOCAL_DATA_PATH}/{self.path}/{wav_id}.wav"

        # corresponding target value
        label = self.metadata.iloc[idx]["compatible"] # Changed to compatible
        category = category_dict[label]

        # load in the audio
        audio, sr = librosa.load(wav_path, sr=16000)

        if self.transform:
            audio = self.transform(audio)

        audio = torch.tensor(audio, dtype=torch.float32)
        category = torch.tensor(category, dtype=torch.long)

        return audio.to(device), category.to(device), label

In [4]:
'''
spectogram_transform from lecture notes

- Extended with help from AI to either pad or crop audio length so that audio files are uniform length
'''
def spectrogram_transform(y):
    mel_spectrogram = librosa.feature.melspectrogram(
        y=y,
        sr=16000,
        n_mels=128,
        fmax=8000
    )
    mel_spect = librosa.power_to_db(
        mel_spectrogram,
        ref=np.max
    )
    current_length = mel_spect.shape[1]

    # If audio shorter than MAX_LENGTH, use padding to extend length
    if current_length < MAX_LENGTH:

        padded = np.full(
            (128, MAX_LENGTH),
            -80,
            dtype=np.float32
        )
        padded[:, :current_length] = mel_spect
        mel_spect = padded

    # If audio longer than MAX_LENGTH, crop to shorten length
    else:
        mel_spect = mel_spect[:, :MAX_LENGTH]

    return mel_spect

In [5]:
'''
transform_pipeline from lecture notes

- shift audio for additional training data
'''
def transform_pipeline(y):

    # previous transform to obtain spectrogram
    spec = spectrogram_transform(y)

    # randomly translate the spectrogram by up to 20% of its width in either direction and pad with -80 dB (the minimum value in the spectrogram) on the side that gets rolled over
    max_translation_frac = 0.2
    translation_frac = np.random.uniform(0,max_translation_frac)
    pixels_to_translate = int(translation_frac * spec.shape[1])
    pixels_to_translate = np.random.choice([-pixels_to_translate, pixels_to_translate])
    translated_spec = np.roll(spec, shift=pixels_to_translate, axis=1)
    if pixels_to_translate > 0:
        translated_spec[:, :pixels_to_translate] = -80
    else:
        translated_spec[:, pixels_to_translate:] = -80
    return translated_spec

In [6]:
'''
ConvNet class from lecture notes
'''
class ConvNet(torch.nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.pipeline = torch.nn.Sequential(
            torch.nn.Conv2d(1, 16, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(16, 32, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(32, 64, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(64, 128, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
            torch.nn.Flatten(),
            torch.nn.Linear(9216, num_classes)
        )

    def forward(self, x):
        out = x.unsqueeze(1)
        return self.pipeline(out)

In [7]:
'''
evaluate function from lecture notes
'''
def evaluate(model, data_loader):
    confusion_matrix = torch.zeros(len(CATEGORIES), len(CATEGORIES), dtype=torch.int32)
    loss_fn = torch.nn.CrossEntropyLoss()

    total_loss = 0

    model.eval() # AI recommended
    with torch.no_grad(): # AI recommended addition of this line
        for X, y, labels in data_loader:
            pred = model(X)
            total_loss += loss_fn(pred, y).item()
            y_pred = torch.argmax(pred, dim=1)
            for true_label, pred_label in zip(y, y_pred):
                confusion_matrix[true_label,pred_label] += 1
    acc = (torch.diag(confusion_matrix).sum() / confusion_matrix.sum()) # Move outside for loop?

    return acc.item(), total_loss, confusion_matrix

In [8]:
'''
predict_bird_species function

input: wav_path (string for datapath to wav file)
output: predicted species (string, species name category using reverse dictionary)
'''
def predict_bird_species(wav_path):
    model.eval()

    with torch.no_grad():
        audio, sr = librosa.load(wav_path, sr=16000)
        spec = spectrogram_transform(audio)

        X = torch.tensor(spec, dtype=torch.float32).unsqueeze(0).to(device)

        # Probability predictions of categories
        outputs = model(X)

        # Most likely category according to the model = prediction
        pred = torch.argmax(outputs, dim=1)

        return reverse_category_dict[pred.item()]

In [9]:
'''
NEW WAV

- Combines folder organization concepts from lecture notes with miniproject 3 download function notes
- AI used to debug
'''
def download_wav_data(wav_id, data_dir):

    # Create new folder for data directory if doesn't already exist
    if not os.path.exists(f"{LOCAL_DATA_PATH}/{data_dir}"):
        os.makedirs(f"{LOCAL_DATA_PATH}/{data_dir}")

    destination = f"{LOCAL_DATA_PATH}/{data_dir}/{wav_id}.wav" # Destination for specific audio track (will be converted to .wav before saved)

    # AI helped with the following mp3 to wav conversion.
    if not os.path.exists(destination):

        url = f"{DATA_URL}/mp3/{wav_id}.mp3"

        response = requests.get(url)

        if response.status_code == 200:
            audio = AudioSegment.from_file(
                io.BytesIO(response.content),
                format="mp3"
            )
            audio.export(destination, format="wav") # Save .wav file

        else:
            print(f"Failed download: {wav_id}")

In [10]:
import subprocess
'''
Written by AI, as I was learning to use ffmpeg methods
'''
def get_local_duration(file_path):
    command = [
        'ffprobe',
        '-v', 'error',
        '-show_entries', 'format=duration',
        '-of', 'default=noprint_wrappers=1:nokey=1',
        file_path
    ]

    result = subprocess.run(command, capture_output=True, text=True)
    return float(result.stdout.strip())

In [11]:
import subprocess
import os

def combine_files(file_list, output_name):
    # 1. Create a temporary text file listing the clips
    with open("inputs.txt", "w") as f:
        for file in file_list:
            f.write(f"file '{file}'\n")

    # 2. Run the concat command
    command = [
        'ffmpeg',
        '-y',
        '-f', 'concat',
        '-safe', '0',
        '-i', 'inputs.txt',
        '-c', 'copy', # This 'copy' mode is lightning fast as it doesn't re-encode
        output_name
    ]
    subprocess.run(command)
    # 3. Clean up the temp file
    os.remove("inputs.txt")

# Usage
# combine_files(["clip1.wav", "clip2.wav"], "combined_song.wav")

In [12]:
def download_segment(file_name, start):
    destination = f"{PREPROCESS}/{songPair}.wav"
    duration = "15" # Seconds
    # This command tells ffmpeg to seek remotely and only download the segment
    command = [
        'ffmpeg',
        '-ss', start, # HH:MM:SS
        '-t', duration,
        '-i', file_name,
        '-acodec', 'pcm_s16le', # Convert to WAV codec
        destination
    ]
    subprocess.run(command)

In [ ]:
from google.colab import drive
drive.mount('/content/drive/Junior/sem2/BeatNet')

In [13]:
'''
Main function
'''
if __name__ == "__main__":
    warnings.filterwarnings("ignore")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Running on {device}")

    metadata = pd.DataFrame(columns=['song', 'compatible'])
    LOCAL_DATA_PATH = "songData_data"
    PREPROCESS = "preprocess_data"
    # Download and pair compatible songs

    # AI GENERATED
    all_songs_paths = []
    for root, dirs, files in os.walk("mixes"):
        for file in files:
            if file.endswith(".mp3"):
                all_songs_paths.append(os.path.join(root, file))

    songPair = 0
    for mix in os.listdir("mixes"):
        mix_path = os.path.join("mixes", mix)
        if not os.path.isdir(mix_path): continue

        songs = sorted(os.listdir(mix_path))
        for song_name, next_song_name in zip(songs, songs[1:]): # zip function for pairing songs
            song_path = os.path.join(mix_path, song_name)
            next_song_path = os.path.join(mix_path, next_song_name)
            # ADD NEXT SONG PATH INITIALIZATION!
            ### NEGATIVE:
            rand_song_path = random.choice(all_songs_paths)

            while rand_song_path == song_path or rand_song_path == next_song_path:
                  rand_song_path = random.choice(all_songs_paths)

            negDestination = f"{LOCAL_DATA_PATH}/NEGATIVE{songPair}.mp3" # Get rid of mix path, so just one folder
            if not os.path.exists(negDestination):
                  # SONG ONE
                  download_segment(song_path, get_local_duration(song_path) - 15) # end of song 1
                  # SONG TWO
                  download_segment(rand_song_path, 0) # start of song 2

                  file_list = [song_path, rand_song_path]
                  output_name = f"{PREPROCESS}/{songPair}.mp3"
                  combine_files(file_list, output_name)
                  metadata.loc[len(metadata)] = [songPair, 0]
            ###
            songPair+=1

            ### POSITIVE
            if next_song:
                destination = f"{PREPROCESS}/{songPair}.wav" # Get rid of /mix so just one folder
                if not os.path.exists(destination): # If it doesnt already exist
                      # SONG ONE
                      download_segment(song_path, get_local_duration(song_path) - 15) # end of song 1
                      # SONG TWO
                      download_segment(next_song_path, 0) # start of song 2

                      file_list = [song_path, next_song_path]
                      output_name = destination
                      combine_files(file_list, output_name)

                      os.remove(song_path)
                      os.remove(next_song_path)
                      metadata.loc[len(metadata)] = [songPair, 1]

                else:
                      print(f"Failed download: {song}")
            songPair += 1



    # Pair and save negative compatible songs
    for mix in os.listdir("mixes"):
        for song, next_song in zip(os.listdir(f"mixes/{mix}")):

            randSong = random.choice(os.listdir(f"mixes/{mix}"))
            while randSong == song or randSong == next_song:
                randSong = random.choice(os.listdir(f"mixes/{mix}"))

            destination = f"{LOCAL_DATA_PATH}/NEGATIVE{songPair}.mp3" # Get rid of mix path, so just one folder
            if not os.path.exists(destination):
                  # SONG ONE
                  download_segment(song, get_local_duration(song) - 15) # end of song 1
                  # SONG TWO
                  download_segment(next_song, 0) # start of song 2

                  file_list = [song, next_song]
                  output_name = f"{PREPROCESS}/{songPair}.mp3"
                  combine_files(file_list, output_name)

                  os.remove(song)
                  os.remove(next_song)
                  metadata.loc[songPair] = [songPair, 1]
            else:
                  print(f"Failed download: {song}")
        songPair += 1

    # Train test split (useful for testing stages of project, although splitting is still ok for 50% accuracy)
    # Split evenly around compatible??
    train_df, test_df = train_test_split(
        metadata,
        test_size=0.05, # Originally 0.2. Ideally 0.0, but 0.05 should be sufficient training data for greater than 50% accuracy
        random_state=48,
        stratify=data["compatible"], # Added
        shuffle=True
    )

    '''
    Initialize CATEGORIES variable and dictionaries for switching between category and category index
    '''
    CATEGORIES = metadata["compatible"].unique().tolist() # Use CATEGORIES list to work with lecture note code.

    category_dict = {
        category: idx
        for idx, category in enumerate(CATEGORIES)
    }

    reverse_category_dict = { # Used when classifying unknown files
        idx: category
        for category, idx in category_dict.items()
    }


    '''
    Download all the songs as wav files in their respective folders (train, test, and unknown folders)

    train_df["id"].apply(lambda x: download_wav_data(x, "train"))
    test_df["id"].apply(lambda x: download_wav_data(x, "test"))
    '''

    # INSTEAD, MOVE test and train into different folders, hopefully a way to do this efficiently
    import shutil
    '''
    for song in train_df:
        shutil.move(PREPROCESS + song, LOCAL_DATA_PATH + '/train/' + song)
    for song in test_df:
        shutil.move(PREPROCESS + song, LOCAL_DATA_PATH + '/test/' + song)
    '''
    # AI debugged version:
    for folder in ['train', 'test']:
        os.makedirs(os.path.join(LOCAL_DATA_PATH, folder), exist_ok=True)
    for _, row in train_df.iterrows():
        shutil.move(f"{PREPROCESS}/{row['id']}.wav", f"{LOCAL_DATA_PATH}/train/{row['id']}.wav")
    for _, row in test_df.iterrows():
        shutil.move(f"{PREPROCESS}/{row['id']}.wav", f"{LOCAL_DATA_PATH}/test/{row['id']}.wav")

    # INCREASE
    MAX_LENGTH = 157 # CHANGE?

    '''
    Initialize dataloaders for training and testing data
    '''
    # First initialize WavDatasets for use in DataLoaders
    spectrogram_train_dataset = WavDataset(train_df, "train", transform=transform_pipeline)
    spectrogram_test_dataset = WavDataset(test_df, "test", transform=spectrogram_transform)

    spectrogram_train_loader = DataLoader(spectrogram_train_dataset, batch_size=8, shuffle=True)
    spectrogram_test_loader = DataLoader(spectrogram_test_dataset,batch_size=8,shuffle=False)

    '''
    Initialize model, loss function, and optimizer, then train model (from lecture notes)
    '''
    model = ConvNet(num_classes=len(CATEGORIES)).to(device)

    loss_fn = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(10):
        model.train()
        for X_batch, y_batch, _ in spectrogram_train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = loss_fn(outputs, y_batch)
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch+1}: {loss.item():.4f}") # epoch loss updates

    '''
    MAKE THE FINAL PREDICTIONS
    '''

Running on cpu


FileNotFoundError: [Errno 2] No such file or directory: 'mixes'